<a href="https://colab.research.google.com/github/Nourimanekerroumi996/olist-pyspark/blob/seller-performance/notebooks/olist_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Olist E-commerce Analysis with PySpark

End-to-end analysis of the Olist Brazilian e-commerce dataset (~100K orders, 2016–2018):
1. **Data quality & cleaning**
2. **Seller performance & delivery analysis**
3. **Customer loyalty analytics**

#  Part 1: Data Quality & Cleaning

Before any analysis, I assess the quality of the Olist dataset: identifier uniqueness,
order statuses, missing values, column types and value consistency.

In [ ]:
#  Setup : installation, données et Spark (à relancer à chaque nouvelle session)
!pip install pyspark kaggle -q

import os
from google.colab import files
if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
    files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d olistbr/brazilian-ecommerce -q
!unzip -o -q brazilian-ecommerce.zip -d olist_data

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark = (SparkSession.builder.appName("Olist")
         .config("spark.sql.shuffle.partitions", "8").getOrCreate())

dfs = {}
for f in sorted(os.listdir("olist_data")):
    name = f.replace(".csv", "").replace("olist_", "").replace("_dataset", "")
    dfs[name] = spark.read.csv(f"olist_data/{f}", header=True, inferSchema=True,
                               multiLine=True, escape='"')
print("Tables chargées :", list(dfs.keys()))

Dataset URL: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
License(s): CC-BY-NC-SA-4.0
Tables chargées : ['customers', 'geolocation', 'order_items', 'order_payments', 'order_reviews', 'orders', 'products', 'sellers', 'product_category_name_translation']


## 0. Tables Overview

Before cleaning anything, I review the 9 tables of the Olist dataset:
their size (rows × columns) and their columns with data types.

### 0.1 Table sizes

In [ ]:
import pandas as pd

# Pour chaque table : son nom, son nombre de lignes et de colonnes
resume = []
for name, df in dfs.items():
    resume.append({
        "table": name,
        "lignes": df.count(),
        "colonnes": len(df.columns),
    })

# Tableau pandas uniquement pour un affichage plus lisible
resume_df = pd.DataFrame(resume).sort_values("lignes", ascending=False)
display(resume_df)

,table,lignes,colonnes
1,geolocation,1000163,5
2,order_items,112650,7
3,order_payments,103886,5
0,customers,99441,5
5,orders,99441,8
4,order_reviews,99224,7
6,products,32951,9
7,sellers,3095,4
8,product_category_name_translation,71,2


**Observations:**
- `geolocation` is by far the largest table (~1M rows): the same zip code prefix appears many times with different GPS coordinates.
- `orders` and `customers` have exactly the same number of rows (99,441): a customer record is created for every order.
- `order_items` (~112K rows) is larger than `orders`: an order can contain several items.

### 0.2 Columns and data types
for each table: a list of its columns and the type detected by SPARK.


In [ ]:
for name, df in dfs.items():
    print(f"\n📋 {name.upper()}  ({df.count():,} lignes × {len(df.columns)} colonnes)")
    print("-" * 60)
    # df.dtypes donne une liste de paires (nom_colonne, type)
    for col_name, col_type in df.dtypes:
        print(f"   {col_name:40s} {col_type}")


📋 CUSTOMERS  (99,441 lignes × 5 colonnes)
------------------------------------------------------------
   customer_id                              string
   customer_unique_id                       string
   customer_zip_code_prefix                 int
   customer_city                            string
   customer_state                           string

📋 GEOLOCATION  (1,000,163 lignes × 5 colonnes)
------------------------------------------------------------
   geolocation_zip_code_prefix              int
   geolocation_lat                          double
   geolocation_lng                          double
   geolocation_city                         string
   geolocation_state                        string

📋 ORDER_ITEMS  (112,650 lignes × 7 colonnes)
------------------------------------------------------------
   order_id                                 string
   order_item_id                            int
   product_id                               string
   seller_id              

**Observations:**
- Dates are correctly parsed as `timestamp`, which allows delivery delays to be computed directly.
- Identifiers (`order_id`, `customer_id`, `product_id`…) are `string`: they are codes, not numbers.
- Product weights and dimensions are `int`: they will be cast to `float` later in the cleaning process.
- Two `products` columns contain a typo from the source data (`product_name_lenght`, `product_description_lenght`).

## 1. Customer Identifiers Check
check if costumer_id and customer_unique_id are unique ids.



In [ ]:


customers = dfs["customers"]

# Nombre total de lignes dans la table
total = customers.count()

# Nombre de valeurs différentes pour chaque identifiant
nb_customer_id = customers.select("customer_id").distinct().count()
nb_unique_id   = customers.select("customer_unique_id").distinct().count()

print(f"Lignes totales             : {total:,}")
print(f"customer_id distincts      : {nb_customer_id:,}  -> unique ? {nb_customer_id == total}")
print(f"customer_unique_id distincts : {nb_unique_id:,}  -> unique ? {nb_unique_id == total}")


Lignes totales             : 99,441
customer_id distincts      : 99,441  -> unique ? True
customer_unique_id distincts : 96,096  -> unique ? False


**Conclusion:**
- `customer_id` is unique: it is the table's key, but a new one is generated **for every order**.
- `customer_unique_id` is not unique: 99,441 rows for 96,096 real customers.
  About 3,300 rows belong to returning customers .

## 2. Surrogate Key

Goal: add a simple incremental integer key (1, 2, 3, …) to the `customers` table.

In [ ]:
from pyspark.sql.window import Window

# 1. Define the window: the whole table, sorted by customer_id
w = Window.orderBy("customer_id")

# 2. Number each row according to its position in the window: 1, 2, 3, ...
customers_seq = customers.withColumn("id", F.row_number().over(w))

# 3. Put the new id column first for readability
customers_seq = customers_seq.select("id", *customers.columns)

customers_seq.show(5, truncate=False)

+---+--------------------------------+--------------------------------+------------------------+-------------+--------------+
|id |customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city|customer_state|
+---+--------------------------------+--------------------------------+------------------------+-------------+--------------+
|1  |00012a2ce6f8dcda20d059ce98491703|248ffe10d632bebe4f7267f1f44844c9|6273                    |osasco       |SP            |
|2  |000161a058600d5901f007fab4c27140|b0015e09bb4b6e47c52844fab5fb6638|35550                   |itapecerica  |MG            |
|3  |0001fd6190edaaf884bcaf3d49edf079|94b11d37cd61cb2994a194d11f89682b|29830                   |nova venecia |ES            |
|4  |0002414f95344307404f0ace7a26f1d5|4893ad4ea28b2c5b3ddf4e82e79db9e6|39664                   |mendonca     |MG            |
|5  |000379cdec625522490c315e70c7a9fb|0b83f73b19c2019e182fd552c048a22c|4841                    |sao paulo    |SP      

## 3. Delivered vs Non-Delivered Orders

Goal: keep only delivered orders for downstream analysis (delivery times, revenue),
while storing the other orders in a separate dataset instead of discarding them.

In [ ]:
orders = dfs["orders"]

# Distribution of order statuses
orders.groupBy("order_status").count().orderBy(F.desc("count")).show()

+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|96478|
|     shipped| 1107|
|    canceled|  625|
| unavailable|  609|
|    invoiced|  314|
|  processing|  301|
|     created|    5|
|    approved|    2|
+------------+-----+



In [ ]:
is_delivered = F.col("order_status") == "delivered"

orders_delivered     = orders.filter(is_delivered)
orders_not_delivered = orders.filter(~is_delivered | F.col("order_status").isNull())

n_total = orders.count()
n_deliv = orders_delivered.count()
n_other = orders_not_delivered.count()

print(f"Total orders        : {n_total:,}")
print(f"Delivered           : {n_deliv:,}  ({n_deliv / n_total:.1%})")
print(f"Not delivered       : {n_other:,}  ({n_other / n_total:.1%})")
print(f"Check (sum = total) : {n_deliv + n_other == n_total}")

Total orders        : 99,441
Delivered           : 96,478  (97.0%)
Not delivered       : 2,963  (3.0%)
Check (sum = total) : True


In [ ]:
# What is inside the non-delivered dataframe?
orders_not_delivered.groupBy("order_status").count().orderBy(F.desc("count")).show()

+------------+-----+
|order_status|count|
+------------+-----+
|     shipped| 1107|
|    canceled|  625|
| unavailable|  609|
|    invoiced|  314|
|  processing|  301|
|     created|    5|
|    approved|    2|
+------------+-----+



**Conclusion:**
- 96,478 orders (97.0%) are `delivered` and kept in `orders_delivered` for the next steps.
- The remaining 2,963 orders (3.0%) are stored in `orders_not_delivered`:
  mostly `shipped` (1,107), `canceled` (625) and `unavailable` (609).
- Sanity check: delivered + non-delivered = total, so no rows were lost.

## 4. Missing Approval Dates
Remove orders with no approval date since an order without a validated payement connot be used to analyze the order lifecyle.


In [ ]:
# How many approval dates are missing, and for which statuses?
print("Null approval dates in all orders      :", orders.filter(F.col("order_approved_at").isNull()).count())
print("Null approval dates in delivered orders:", orders_delivered.filter(F.col("order_approved_at").isNull()).count())

(orders.filter(F.col("order_approved_at").isNull())
       .groupBy("order_status")
       .count()
       .orderBy(F.desc("count"))
       .show())

Null approval dates in all orders      : 160
Null approval dates in delivered orders: 14
+------------+-----+
|order_status|count|
+------------+-----+
|    canceled|  141|
|   delivered|   14|
|     created|    5|
+------------+-----+



In [ ]:
# Remove rows where order_approved_at is null
orders_clean = orders_delivered.dropna(subset=["order_approved_at"])

print(f"Before : {orders_delivered.count():,}")
print(f"After  : {orders_clean.count():,}")
print(f"Removed: {orders_delivered.count() - orders_clean.count():,}")

# Check: no null approval date left
print("Remaining nulls:", orders_clean.filter(F.col("order_approved_at").isNull()).count())

Before : 96,478
After  : 96,464
Removed: 14
Remaining nulls: 0


**Conclusion:**
- 160 orders in the full table have no approval date, mostly `canceled` orders (never validated).
- Among delivered orders, only 14 are affected: these are likely data-quality issues
  (delivered without a recorded approval), so they are removed.
- `orders_clean` now contains delivered orders with a valid approval date.

## 5. Number of Unique Customers

Count the number of real customers. As shown in section 1, `customer_id` is generated
for every order, so real customers must be counted with `customer_unique_id`.

In [ ]:
n_unique_customers = customers.agg(F.countDistinct("customer_unique_id")).first()[0]

print(f"Unique customers : {n_unique_customers:,}")
print(f"Total orders     : {customers.count():,}")

Unique customers : 96,096
Total orders     : 99,441


## 6. Geographic Coverage
count the number of distinct zip code prefixes in the `geolocation` table.

In [ ]:
geolocation = dfs["geolocation"]

n_rows = geolocation.count()
n_zip  = geolocation.agg(F.countDistinct("geolocation_zip_code_prefix")).first()[0]

print(f"Rows in geolocation      : {n_rows:,}")
print(f"Distinct zip code prefix : {n_zip:,}")
print(f"Average rows per zip     : {n_rows / n_zip:.1f}")

Rows in geolocation      : 1,000,163
Distinct zip code prefix : 19,015
Average rows per zip     : 52.6


**Conclusion:**
- The `geolocation` table has ~1M rows but only **19,015 distinct zip code prefixes**,
  which is about 53 GPS points per zip code on average.


## 7. Product Dimensions Typing

cast product weight and dimension columns from integer to float,
so that downstream calculations (volume, weight categories, averages) are done on decimal values.

In [ ]:
products = dfs["products"]
dim_cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]

# Cast all dimension columns to float
products = products.withColumns({c: F.col(c).cast("float") for c in dim_cols})

products.select(dim_cols).printSchema()
products.select(dim_cols).show(3)

root
 |-- product_weight_g: float (nullable = true)
 |-- product_length_cm: float (nullable = true)
 |-- product_height_cm: float (nullable = true)
 |-- product_width_cm: float (nullable = true)

+----------------+-----------------+-----------------+----------------+
|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+----------------+-----------------+-----------------+----------------+
|           225.0|             16.0|             10.0|            14.0|
|          1000.0|             30.0|             18.0|            20.0|
|           154.0|             18.0|              9.0|            15.0|
+----------------+-----------------+-----------------+----------------+
only showing top 3 rows


**Conclusion:**
- The 4 weight and dimension columns are now `float`.
- The cast did not create new missing values: the only nulls (2 products) already existed
  in the source data (products without any physical information).

## 8. Weight Categories

classify products by weight (stored in grams): Light (< 5 kg), Medium (5–10 kg),
Heavy (10–20 kg) and Very heavy (> 20 kg).

In [ ]:
# First, check products with no weight
products.filter(F.col("product_weight_g").isNull()) \
        .select("product_id", "product_category_name", *dim_cols) \
        .show(truncate=False)

+--------------------------------+---------------------+----------------+-----------------+-----------------+----------------+
|product_id                      |product_category_name|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------------------+---------------------+----------------+-----------------+-----------------+----------------+
|09ff539a621711667c43eba6a3bd8466|bebes                |NULL            |NULL             |NULL             |NULL            |
|5eb564652db742ff8f28759cd8d2652a|NULL                 |NULL            |NULL             |NULL             |NULL            |
+--------------------------------+---------------------+----------------+-----------------+-----------------+----------------+



In [ ]:
w = F.col("product_weight_g")

products = products.withColumn("weight_category",
    F.when(w.isNull(), "Unknown")
     .when(w < 5000,   "Light")
     .when(w < 10000,  "Medium")
     .when(w <= 20000, "Heavy")
     .otherwise("Very heavy")
)

products.groupBy("weight_category").count().orderBy(F.desc("count")).show()

+---------------+-----+
|weight_category|count|
+---------------+-----+
|          Light|28752|
|         Medium| 2291|
|          Heavy| 1421|
|     Very heavy|  485|
|        Unknown|    2|
+---------------+-----+



**Conclusion:**
- 2 products have no weight in the source data: they are labeled `Unknown`
- Most products are Light (< 5 kg); heavy and very heavy products are a small minority.

## 9. Most Frequent Category Keywords

find the 5 most frequent keywords in product category names counted over all products.

In [ ]:
stop_words = ["e", "de", "do", "da", "dos", "das"]   # Portuguese linking words

keywords = (products
    .select(F.explode(F.split("product_category_name", "_")).alias("keyword"))
    .filter(F.col("keyword").isNotNull() & ~F.col("keyword").isin(stop_words))
    .groupBy("keyword").count()
    .orderBy(F.desc("count"))
)

keywords.show(5)

+-------+-----+
|keyword|count|
+-------+-----+
| moveis| 3271|
|   mesa| 3029|
|   cama| 3029|
|  banho| 3029|
|esporte| 2886|
+-------+-----+
only showing top 5 rows


In [ ]:
# Number of different keywords (one row per keyword in the "keywords" table)
n_distinct_keywords = keywords.count()

# Total number of keyword occurrences across all products (sum of the counts)
n_total_occurrences = keywords.agg(F.sum("count")).first()[0]

print(f"Distinct keywords  : {n_distinct_keywords:,}")
print(f"Total occurrences  : {n_total_occurrences:,}")

Distinct keywords  : 107
Total occurrences  : 61,713


**Conclusion:** the most frequent keywords reflect Olist's main product families
(home, furniture, health & beauty, sports…). Portuguese linking words (`e`, `de`…) were excluded.

## 10. Product Volume vs Weight

find the 10 largest products by volume (length × width × height, in cm³)
and check whether they are also the heaviest.

In [ ]:
from pyspark.sql.window import Window

# 1. Volume in cm³
products = products.withColumn("volume_cm3",
    F.col("product_length_cm") * F.col("product_width_cm") * F.col("product_height_cm"))

# 2. Weight rank among all products (1 = heaviest)
products = products.withColumn("weight_rank",
    F.rank().over(Window.orderBy(F.desc("product_weight_g"))))

# 3. Top 10 by volume, with their weight rank
top_volume = products.orderBy(F.desc("volume_cm3")).limit(10)

top_volume.select("product_id", "product_category_name", "product_weight_g" ,
                  "volume_cm3", "weight_rank").show(truncate=False)

+--------------------------------+---------------------+----------------+----------+-----------+
|product_id                      |product_category_name|product_weight_g|volume_cm3|weight_rank|
+--------------------------------+---------------------+----------------+----------+-----------+
|256a9c364b75753b97bee410c9491ad8|utilidades_domesticas|25250.0         |296208.0  |237        |
|f227e2d44f10f7dad30fb4dfa839e7a2|moveis_sala          |30000.0         |294000.0  |2          |
|c1e0531cb1864fd3a0cae57dca55ca80|moveis_sala          |30000.0         |294000.0  |2          |
|3eb14e65e4208c6d94b7a32e41add538|moveis_sala          |30000.0         |294000.0  |2          |
|0b48eade13cfad433122f23739a66898|moveis_decoracao     |14000.0         |294000.0  |1107       |
|90c1b4e040d1d1c45897ec2dad4a809d|moveis_sala          |30000.0         |293706.0  |2          |
|c6fdec160d0f8f488d9041316c85051d|moveis_sala          |30000.0         |288000.0  |2          |
|8d6f2c3454002d3f5aa7479a7fad7

In [ ]:
# How many of the 10 largest products are also among the 10 heaviest?
n_overlap = top_volume.filter(F.col("weight_rank") <= 10).count()
print(f"Largest products that are also in the top 10 heaviest: {n_overlap} / 10")



Largest products that are also in the top 10 heaviest: 8 / 10


**Conclusion:**
- 8 of the 10 largest products are also among the heaviest
- Two exceptions: the largest product overall weighs 25.25 kg (237th heaviest), and a bulky
  decoration item weighs only 14 kg (1,107th heaviest).


## 11. Price Validation

make sure no item has a negative price  in `order_items`,

In [ ]:
order_items = dfs["order_items"]

n_negative = order_items.filter(F.col("price") < 0).count()

print(f"Items with a negative price: {n_negative}")

Items with a negative price: 0


**Conclusion:**
- No negative price nor negative freight value: `order_items` passes this quality check.
